In [1]:
#importing basic libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.cm as cs
from warnings import filterwarnings


In [2]:
df=pd.read_csv('../data/DataCoSupplyChainDataset.csv',encoding='latin-1')

In [3]:
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Product Category Id',
       'Product De

In [4]:
print('rows,cols', df.shape)
print('\ncoloumns:')
print(df.columns.tolist())
print('\nNum duplicates:',df.duplicated().sum())
print('\nMissing values(top 20):')
print(df.isna().sum().sort_values(ascending=False).head(20))

rows,cols (180519, 53)

coloumns:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name', 'Product Price', 'Product

In [5]:
df['Benefit per order']==df['Order Profit Per Order']

0         True
1         True
2         True
3         True
4         True
          ... 
180514    True
180515    True
180516    True
180517    True
180518    True
Length: 180519, dtype: bool

In [6]:
(df['Benefit per order']==df['Order Profit Per Order']).value_counts()

True    180519
Name: count, dtype: int64

In [7]:
(df['Product Status']).value_counts()

Product Status
0    180519
Name: count, dtype: int64

In [8]:
#Data Cleaning
columns_to_drop=[
    'Product Description',
    'Order Zipcode',
    'Customer Lname',
    'Customer Zipcode',
    'Customer Fname',
    'Customer Email',
    'Customer Password',
    'Customer Street',
    'Order Profit Per Order', #same as Benifit Per order
    'Longitude',
    'Latitude',
    'Order Item Cardprod Id',
    'Order Item Id',
    'Order Item Discount',
    'Order Item Discount Rate',
    'Order Item Product Price',
    'Category Id',
    'Department Id',
    'Order Id',
    'Order Customer Id',
    'Customer Id',
    'Product Card Id',
    'Product Category Id',
    'Product Status', #has only single value
    'Customer City',
    'Order City',
    'Order Country',
    'Order State',
    'Customer State',
    'Market'
]

In [9]:
len(df)

180519

In [10]:
#checking the current 
print('rows, cols:',df.shape)
print('\nMissing values(top 5):')
print(df.isna().sum().sort_values(ascending=False).head(5))

rows, cols: (180519, 53)

Missing values(top 5):
Product Description    180519
Order Zipcode          155679
Customer Lname              8
Customer Zipcode            3
Type                        0
dtype: int64


In [11]:
df.shape

(180519, 53)

In [ ]:
#Filtering out the columns that have missing values, not important, or have only single value.
df=df.drop(columns=columns_to_drop)

#Filtering out the cancelled orders because they are not relevant for delivery time analysis.
df=df[df['Delivery Status']!='Shipping Canceled']

In [13]:
#Date conversion
for i in ['order date (DateOrders)','shipping date (DateOrders)']:
    df[i]=pd.to_datetime(df[i], errors='coerce',dayfirst=False)

In [14]:
#checking the overview again after the first bit of cleaning
print('rows, cols:',df.shape)
print('\nMissing values(top 5):')
print(df.isna().sum().sort_values(ascending=False).head(5))

rows, cols: (180519, 23)

Missing values(top 5):
Type                          0
Order Item Profit Ratio       0
shipping date (DateOrders)    0
Product Price                 0
Product Name                  0
dtype: int64


In [15]:
#No of values in category columns with low cardinality
for col in df.columns:
    if df[col].nunique()<10:
        print(f'\n{col} value counts:')
        print(df[col].value_counts())


Type value counts:
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

Days for shipping (real) value counts:
Days for shipping (real)
2    56618
3    28765
6    28723
4    28513
5    28163
0     5080
1     4657
Name: count, dtype: int64

Days for shipment (scheduled) value counts:
Days for shipment (scheduled)
4    107752
2     35216
1     27814
0      9737
Name: count, dtype: int64

Delivery Status value counts:
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Late_delivery_risk value counts:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Customer Country value counts:
Customer Country
EE. UU.        111146
Puerto Rico     69373
Name: count, dtype: int64

Customer Segment value counts:
Customer Segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64

Order Item Quantity 

In [17]:
#Calculation of order processing time and delay
df['Order Processing Time']=(df['shipping date (DateOrders)']-df['order date (DateOrders)']).dt.days
df['Delay']=df['Order Processing Time']-df['Days for shipment (scheduled)']
df['Is_Delayed']=df['Delay']>0
df['Order Month']=df['order date (DateOrders)'].dt.month
df['Order_day']=df['order date (DateOrders)'].dt.day_name()
df['Order_hour']=df['order date (DateOrders)'].dt.hour

In [19]:
df.describe()

,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Late_delivery_risk,order date (DateOrders),Order Item Profit Ratio,Order Item Quantity,Sales,Order Item Total,Product Price,shipping date (DateOrders),Order Processing Time,Delay,Order Month,Order_hour
count,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519,180519.000000,180519.000000,180519.000000,180519.000000,180519.000000,180519,180519.000000,180519.000000,180519.000000,180519.000000
mean,3.497654,2.931847,21.974989,183.107609,0.548291,2016-06-12 17:47:04.669868,0.120647,2.127638,203.772096,183.107609,141.232550,2016-06-16 05:45:23.202433,3.471856,0.540010,6.235449,11.483689
min,0.000000,0.000000,-4274.979980,7.490000,0.000000,2015-01-01 00:00:00,-2.750000,1.000000,9.990000,7.490000,9.990000,2015-01-03 00:00:00,0.000000,-2.000000,1.000000,0.000000
25%,2.000000,2.000000,7.000000,104.379997,0.000000,2015-09-21 13:49:00,0.080000,1.000000,119.980003,104.379997,50.000000,2015-09-25 06:59:00,2.000000,0.000000,3.000000,5.000000
50%,3.000000,4.000000,31.520000,163.990005,1.000000,2016-06-11 13:06:00,0.270000,1.000000,199.919998,163.990005,59.990002,2016-06-15 08:32:00,3.000000,1.000000,6.000000,11.000000
75%,5.000000,4.000000,64.800003,247.399994,1.000000,2017-03-01 08:42:00,0.360000,3.000000,299.950012,247.399994,199.990005,2017-03-04 21:29:00,5.000000,1.000000,9.000000,17.000000
max,6.000000,4.000000,911.799988,1939.989990,1.000000,2018-01-31 23:38:00,0.500000,5.000000,1999.989990,1939.989990,1999.989990,2018-02-06 22:14:00,6.000000,4.000000,12.000000,23.000000
std,1.623722,1.374449,104.433526,120.043670,0.497664,NaN,0.466796,1.453451,132.273077,120.043670,139.732492,NaN,1.670471,1.491881,3.403571,6.923006
